In [1]:
import numpy as np
import torch
import torch.nn as nn

from sklearn.model_selection import train_test_split

from torch.utils.data import (
    TensorDataset,
    DataLoader
)

In [2]:
latent_vectors = np.load(
    "../extract_latents/latent_vectors_len40_v2.npy"
)

print(latent_vectors.shape)

(7405, 64)


In [3]:
encoded_sequences = np.load(
    "../encoded_sequences_len40_v2.npy"
)

print(encoded_sequences.shape)

(7405, 42)


In [4]:
print(
    latent_vectors.shape[0]
)

print(
    encoded_sequences.shape[0]
)

7405
7405


In [5]:
X_train, X_test, y_train, y_test = train_test_split(
    latent_vectors,
    encoded_sequences,
    test_size=0.1,
    random_state=42
)

print(X_train.shape)
print(X_test.shape)

print(y_train.shape)
print(y_test.shape)

(6664, 64)
(741, 64)
(6664, 42)
(741, 42)


In [6]:
X_train = torch.tensor(
    X_train,
    dtype=torch.float32
)

X_test = torch.tensor(
    X_test,
    dtype=torch.float32
)

y_train = torch.tensor(
    y_train,
    dtype=torch.long
)

y_test = torch.tensor(
    y_test,
    dtype=torch.long
)

In [7]:
train_dataset = TensorDataset(
    X_train,
    y_train
)

test_dataset = TensorDataset(
    X_test,
    y_test
)

In [8]:
train_loader = DataLoader(
    train_dataset,
    batch_size=128,
    shuffle=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=128,
    shuffle=False
)

print(len(train_loader))

53


In [9]:
device = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)

print(device)

cuda


In [10]:
class LatentLSTMDecoder(
    nn.Module
):

    def __init__(
        self,
        latent_dim=64,
        embed_dim=128,
        hidden_dim=256,
        vocab_size=23
    ):

        super().__init__()

        self.embedding = nn.Embedding(
            vocab_size,
            embed_dim,
            padding_idx=0
        )

        self.latent_to_h = nn.Linear(
            latent_dim,
            hidden_dim
        )

        self.latent_to_c = nn.Linear(
            latent_dim,
            hidden_dim
        )

        self.lstm = nn.LSTM(
            embed_dim,
            hidden_dim,
            batch_first=True
        )

        self.output_layer = nn.Linear(
            hidden_dim,
            vocab_size
        )

    def forward(
        self,
        z,
        decoder_input
    ):

        embedded = self.embedding(
            decoder_input
        )

        h0 = self.latent_to_h(
            z
        ).unsqueeze(0)

        c0 = self.latent_to_c(
            z
        ).unsqueeze(0)

        out, _ = self.lstm(
            embedded,
            (h0,c0)
        )

        logits = self.output_layer(
            out
        )

        return logits

In [11]:
model = LatentLSTMDecoder().to(device)

print(
    sum(
        p.numel()
        for p in model.parameters()
    )
)

437399


In [12]:
import torch.nn.functional as F

def decoder_loss(
    logits,
    target
):

    loss = F.cross_entropy(
        logits.reshape(-1,23),
        target.reshape(-1),
        ignore_index=0
    )

    return loss

In [13]:
optimizer = torch.optim.Adam(
    model.parameters(),
    lr=1e-3
)

In [14]:
num_epochs = 100

for epoch in range(num_epochs):

    model.train()

    total_loss = 0

    for latent, seq in train_loader:

        latent = latent.to(device)

        seq = seq.to(device)

        decoder_input = seq[:, :-1]

        target = seq[:, 1:]

        logits = model(
            latent,
            decoder_input
        )

        loss = decoder_loss(
            logits,
            target
        )

        optimizer.zero_grad()

        loss.backward()

        torch.nn.utils.clip_grad_norm_(
            model.parameters(),
            1.0
        )

        optimizer.step()

        total_loss += loss.item()

    print(
        f"Epoch [{epoch+1}/{num_epochs}] "
        f"Loss={total_loss/len(train_loader):.4f}"
    )

Epoch [1/100] Loss=2.7691
Epoch [2/100] Loss=2.5336
Epoch [3/100] Loss=2.3640
Epoch [4/100] Loss=2.2471
Epoch [5/100] Loss=2.1686
Epoch [6/100] Loss=2.0719
Epoch [7/100] Loss=1.9970
Epoch [8/100] Loss=1.9237
Epoch [9/100] Loss=1.8632
Epoch [10/100] Loss=1.8077
Epoch [11/100] Loss=1.7636
Epoch [12/100] Loss=1.7190
Epoch [13/100] Loss=1.6897
Epoch [14/100] Loss=1.6533
Epoch [15/100] Loss=1.6120
Epoch [16/100] Loss=1.5788
Epoch [17/100] Loss=1.5420
Epoch [18/100] Loss=1.5162
Epoch [19/100] Loss=1.4849
Epoch [20/100] Loss=1.4593
Epoch [21/100] Loss=1.4294
Epoch [22/100] Loss=1.4162
Epoch [23/100] Loss=1.3855
Epoch [24/100] Loss=1.3668
Epoch [25/100] Loss=1.3303
Epoch [26/100] Loss=1.3215
Epoch [27/100] Loss=1.2901
Epoch [28/100] Loss=1.2649
Epoch [29/100] Loss=1.2522
Epoch [30/100] Loss=1.2347
Epoch [31/100] Loss=1.2204
Epoch [32/100] Loss=1.1930
Epoch [33/100] Loss=1.1755
Epoch [34/100] Loss=1.1698
Epoch [35/100] Loss=1.1420
Epoch [36/100] Loss=1.1229
Epoch [37/100] Loss=1.1108
Epoch [38/

In [15]:
model.eval()

correct = 0
total = 0

with torch.no_grad():

    for latent, seq in test_loader:

        latent = latent.to(device)

        seq = seq.to(device)

        decoder_input = seq[:, :-1]

        target = seq[:, 1:]

        logits = model(
            latent,
            decoder_input
        )

        pred = logits.argmax(dim=2)

        mask = (target != 0)

        correct += (
            ((pred == target) & mask)
            .sum()
            .item()
        )

        total += (
            mask
            .sum()
            .item()
        )

token_acc = correct / total

print(
    f"Token Accuracy: {token_acc:.4f}"
)

Token Accuracy: 0.6351


In [16]:
model.eval()

latent, seq = next(
    iter(test_loader)
)

latent = latent.to(device)

decoder_input = seq[:, :-1].to(device)

with torch.no_grad():

    logits = model(
        latent,
        decoder_input
    )

pred = logits.argmax(dim=2)

print("INPUT")
print(
    decoder_input[0][:25]
    .cpu()
    .numpy()
)

print("TARGET")
print(
    seq[0][1:26]
    .cpu()
    .numpy()
)

print("PRED")
print(
    pred[0][:25]
    .cpu()
    .numpy()
)

INPUT
[ 1 19 12 12  4  5  4 15 14  8 15 21 20 21 20 15  3  7  4 16  3  2  0  0
  0]
TARGET
[19 12 12  4  5  4 15 14  8 15 21 20 21 20 15  3  7  4 16  3  2  0  0  0
  0]
PRED
[ 4 15 15 15  7  4  7 21 21  3 21  3  4  3  4  3  4  4  4  4 14  2  2  2
  2]


In [17]:
torch.save(
    model.state_dict(),
    "lstm_decoder_len40.pt"
)

In [18]:
torch.save(
    {
        "model_state_dict":
            model.state_dict()
    },
    "lstm_decoder_checkpoint_len40.pt"
)

In [19]:
@torch.no_grad()
def generate_peptide(
    latent,
    max_len=42
):

    generated = [1]  # START

    for _ in range(max_len):

        decoder_input = torch.tensor(
            [generated],
            dtype=torch.long
        ).to(device)

        logits = model(
            latent,
            decoder_input
        )

        next_token = (
            logits[0,-1]
            .argmax()
            .item()
        )

        generated.append(
            next_token
        )

        if next_token == 2:
            break

    return generated

In [20]:
AA_VOCAB = "ACDEFGHIKLMNPQRSTVWY"

idx_to_aa = {}

for i, aa in enumerate(
    AA_VOCAB
):
    idx_to_aa[i+3] = aa

In [21]:
def tokens_to_peptide(
    tokens
):

    peptide = ""

    for token in tokens:

        if token in [0,1]:
            continue

        if token == 2:
            break

        peptide += idx_to_aa.get(
            token,
            ""
        )

    return peptide

In [22]:
import numpy as np

generated_latents = np.load(
    "generated_latents_len40_v2.npy"
)

print(generated_latents.shape)

(10000, 64)


In [23]:
print(generated_latents.shape)

(10000, 64)


In [24]:
all_peptides = []

for i in range(10000):

    z = torch.tensor(
        generated_latents[i:i+1],
        dtype=torch.float32
    ).to(device)

    tokens = generate_peptide(
        z
    )

    peptide = tokens_to_peptide(
        tokens
    )

    all_peptides.append(
        peptide
    )

    print(i)
    print(peptide)
    print(len(peptide))
    print("="*50)

0
KKKKESVAATTLHLTLE
17
1
GLWSTLKNGLGAALGMAQEKG
21
2
SWLRDIWDWIKSLFESDKEWIKNG
24
3
GWWRTFWKKWTGAPLVGHAI
20
4
IIKKLEKLAKGIAEKL
16
5
MKIASEKLHSLKQAFDTW
18
6
ALPIVAKICPNGGKATCPAPIQTRYCSC
28
7
RPKRNRPAMVKLKKPQKL
18
8
KVNGFTLSQWGDYP
14
9
GWLRDFLKKWVGHIANSYK
19
10
GRCTKSCWTGKCHLKLTFTLC
21
11
KWLKKIANILKKA
13
12
GILDILEAAKKSHCIANRIRENTHV
25
13
RLVSTRYNLRIKFWPLTG
18
14
GWLCKVGCWYKTICGVCENKVCYFN
25
15
CFCQTSIIGACHYLKCATGLCKS
23
16
SKQSGRLQMRLKFYGAWSHQPAGH
24
17
GLWEGIKPIVIRWKPKGK
18
18
FNESAFNYSQLIRFAG
16
19
KWKLIKKIASKIKSLA
16
20
CACVKWAPNFVPIECFKPVT
20
21
RWLRLASHGLRRYGRYC
17
22
LQQLLFKRKIRGKR
14
23
GIFGWLICGISRIVL
15
24
GLFTLIKNWKG
11
25
VRPIVSGPRPLVPVGLPWFARAI
23
26
ETFRESLISEGPRLTPQRVKAALGS
25
27
GLMSTWTNLPKKMFSVRG
18
28
ALWGAAANLLVDAPQIAYCRAIYQPAS
27
29
SAIRTFKKSPRKVRLKPSLTR
21
30
GLFNVIKKVASHQLNLAACKAGQ
23
31
CRRKLRWRLSRKIWCLM
17
32
FLPLFAINIKTMCRI
15
33
GIMSSLKKLFKKLAKAKAGK
20
34
CRRKLKQRPGDLQ
13
35
GLLSSVASVAGNFLKNFVKRV
21
36
KYYHKHKHRDHYNLHYL
17
37
GLWQSILRGLKKAIYLGW
18
38
GSILSGFTSILG

In [25]:
unique_peptides = len(
    set(all_peptides)
)

total_peptides = len(
    all_peptides
)

print(
    unique_peptides
)

print(
    total_peptides
)

print(
    unique_peptides /
    total_peptides
)

9999
10000
0.9999


In [26]:
lengths = [
    len(p)
    for p in all_peptides
]

print(
    min(lengths)
)

print(
    max(lengths)
)

print(
    sum(lengths) /
    len(lengths)
)

5
40
18.9075


In [27]:
import pandas as pd

generated_df = pd.DataFrame(
    {
        "Sequence": all_peptides
    }
)

generated_df.to_csv(
    "generated_peptides.csv",
    index=False
)

print(generated_df.shape)

(10000, 1)


In [28]:
import torch
import esm

device = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)

model, alphabet = esm.pretrained.esm2_t33_650M_UR50D()

model = model.to(device)

model.eval()

batch_converter = (
    alphabet.get_batch_converter()
)

print("Loaded")

Loaded


In [29]:
import pandas as pd

generated_df = pd.read_csv(
    "generated_peptides.csv"
)

print(
    generated_df.shape
)

(10000, 1)


In [30]:
import numpy as np
from tqdm import tqdm

sequences = generated_df[
    "Sequence"
].tolist()

batch_size = 32

all_embeddings = []

for i in tqdm(
    range(
        0,
        len(sequences),
        batch_size
    )
):

    batch_seqs = (
        sequences[
            i:i+batch_size
        ]
    )

    batch = [
        (str(j), seq)
        for j, seq
        in enumerate(batch_seqs)
    ]

    _, _, tokens = (
        batch_converter(batch)
    )

    tokens = tokens.to(device)

    with torch.no_grad():

        results = model(
            tokens,
            repr_layers=[33],
            return_contacts=False
        )

    reps = results[
        "representations"
    ][33]

    for j, seq in enumerate(
        batch_seqs
    ):

        seq_len = len(seq)

        emb = (
            reps[
                j,
                1:seq_len+1
            ]
            .mean(0)
            .cpu()
            .numpy()
        )

        all_embeddings.append(
            emb
        )

generated_embeddings = np.array(
    all_embeddings
)

print(
    generated_embeddings.shape
)

100%|█████████████████████████████████████████████████████████████████████████████████| 313/313 [00:19<00:00, 15.78it/s]

(10000, 1280)


In [ ]:
import joblib

amp_model = joblib.load(
    "../../classifers/amp_xgb_classifier.pkl"
)

print(type(amp_model))

<class 'xgboost.sklearn.XGBClassifier'>


In [32]:
amp_probs = amp_model.predict_proba(
    generated_embeddings
)[:,1]

print(
    amp_probs.shape
)

print(
    amp_probs.min()
)

print(
    amp_probs.max()
)

print(
    amp_probs.mean()
)

(10000,)
0.014912674
0.9999201
0.89761716


In [33]:
import pandas as pd

results_df = pd.DataFrame(
    {
        "Sequence": all_peptides,
        "AMP_Probability": amp_probs
    }
)

results_df = results_df.sort_values(
    by="AMP_Probability",
    ascending=False
)

results_df.head()

,Sequence,AMP_Probability
738,RLWRIWKKIKRLW,0.999920
8336,KLLKRLKRLFKKLWKHY,0.999917
6825,LRKLANFFRKKWRKLGKAVLHYL,0.999916
610,LRKRLRWLWNKWKKLG,0.999913
9138,LLKRFWHKLKRLAKGL,0.999912


In [34]:
amp_positive = (
    results_df[
        "AMP_Probability"
    ] >= 0.5
).sum()

print(
    "AMP Positive:",
    amp_positive
)

print(
    "Total:",
    len(results_df)
)

print(
    "Ratio:",
    amp_positive /
    len(results_df)
)

AMP Positive: 9280
Total: 10000
Ratio: 0.928


In [ ]:
import pandas as pd

amp_df = pd.read_csv(
    "../../data/processed_data&code/clean_amp_dataset.csv"
)

train_set = set(
    amp_df["Sequence"]
)

generated_set = set(
    all_peptides
)

novel = [
    p
    for p in generated_set
    if p not in train_set
]

print(
    "Novel:",
    len(novel)
)

print(
    "Generated:",
    len(generated_set)
)

print(
    "Novelity Ratio:",
    len(novel) /
    len(generated_set)
)

Novel: 9997
Generated: 9999
Novelity Ratio: 0.9997999799979999


In [36]:
print(results_df.shape)
print(results_df.columns)

(10000, 2)
Index(['Sequence', 'AMP_Probability'], dtype='object')


In [38]:
short_amp_df = results_df[
    results_df["AMP_Probability"] > 0.5
].copy()

print(short_amp_df.shape)

(9280, 2)


In [39]:
sequences = short_amp_df[
    "Sequence"
].tolist()

print(len(sequences))

9280


In [40]:
import torch
import esm

model, alphabet = esm.pretrained.esm2_t33_650M_UR50D()

batch_converter = (
    alphabet.get_batch_converter()
)

model.eval()

device = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)

model = model.to(device)

In [41]:
data = [
    (str(i), seq)
    for i, seq
    in enumerate(sequences)
]

In [42]:
embeddings = []

for i in range(
    0,
    len(data),
    16
):

    batch = data[
        i:i+16
    ]

    labels, strs, tokens = (
        batch_converter(batch)
    )

    tokens = tokens.to(device)

    with torch.no_grad():

        results = model(
            tokens,
            repr_layers=[33]
        )

    reps = results[
        "representations"
    ][33]

    for j, seq in enumerate(strs):

        emb = reps[
            j,
            1:len(seq)+1
        ].mean(0)

        embeddings.append(
            emb.cpu().numpy()
        )

In [43]:
import numpy as np

embeddings = np.array(
    embeddings
)

print(
    embeddings.shape
)

(9280, 1280)


In [44]:
np.save(
    "embeddings_len40.npy",
    embeddings
)

In [ ]:
import joblib

hemo_model = joblib.load(
    "../../classifers/hemolysis_xgb_classifier.pkl"
)

print(type(hemo_model))

<class 'xgboost.sklearn.XGBClassifier'>


In [46]:
hemo_probs = hemo_model.predict_proba(
    embeddings
)[:,1]

print(hemo_probs.shape)

print(hemo_probs.min())
print(hemo_probs.max())
print(hemo_probs.mean())

(9280,)
0.0007195309
0.9988003
0.24006055


In [54]:
short_amp_df[
    "Hemolysis_Probability"
] = hemo_probs

In [55]:
print((hemo_probs < 0.1).sum())

print((hemo_probs < 0.3).sum())

print((hemo_probs < 0.5).sum())

5061
6680
7393


In [57]:
short_amp_df["Length"] = (
    short_amp_df["Sequence"].str.len()
)

In [58]:
print(short_amp_df.columns)

Index(['Sequence', 'AMP_Probability', 'Hemolysis_Probability', 'Length'], dtype='object')


In [59]:
print(embeddings.shape)

print(hemo_probs.min())
print(hemo_probs.max())
print(hemo_probs.mean())

print((hemo_probs < 0.1).sum())
print((hemo_probs < 0.3).sum())
print((hemo_probs < 0.5).sum())

(9280, 1280)
0.0007195309
0.9988003
0.24006055
5061
6680
7393


In [60]:
short_final_df = short_amp_df[
    (short_amp_df["AMP_Probability"] > 0.9)
    &
    (short_amp_df["Hemolysis_Probability"] < 0.3)
].copy()

print(short_final_df.shape)

(5343, 4)


In [61]:
short_final_df["Final_Score"] = (
    short_final_df["AMP_Probability"]
    -
    short_final_df["Hemolysis_Probability"]
)

short_final_df = short_final_df.sort_values(
    "Final_Score",
    ascending=False
)

short_final_df.head(10)

,Sequence,AMP_Probability,Hemolysis_Probability,Length,Final_Score
5870,KILIKKLKKHIK,0.999444,0.000757,12,0.998687
9773,KILKKIWKKIKQ,0.999614,0.001013,12,0.998601
728,KKLKKFKFLKKAQW,0.999491,0.000973,14,0.998518
8464,KWLKKFKKLKQA,0.999600,0.001430,12,0.998170
5901,KRILKWILKFKK,0.999793,0.001644,12,0.998149
8153,KWLKKIIKKLKK,0.999667,0.001589,12,0.998078
4997,KRKLLHRLKRK,0.999441,0.001406,11,0.998034
882,LLRLLRKKTRKKIW,0.999792,0.001810,14,0.997982
4946,ILKKLWRKIKRK,0.999808,0.001880,12,0.997928
9674,GIIKKLFKKLVSKTAR,0.998970,0.001046,16,0.997923


In [62]:
print(short_final_df.shape)

(5343, 5)


In [63]:
from Bio.SeqUtils.ProtParam import ProteinAnalysis

In [64]:
mw_list = []
pi_list = []
arom_list = []
instab_list = []

In [65]:
for seq in short_final_df["Sequence"]:

    analysis = ProteinAnalysis(seq)

    mw_list.append(
        analysis.molecular_weight()
    )

    pi_list.append(
        analysis.isoelectric_point()
    )

    arom_list.append(
        analysis.aromaticity()
    )

    instab_list.append(
        analysis.instability_index()
    )

In [66]:
short_final_df["MolecularWeight"] = mw_list
short_final_df["pI"] = pi_list
short_final_df["Aromaticity"] = arom_list
short_final_df["InstabilityIndex"] = instab_list

In [67]:
print("Mean Length:",
      short_final_df["Length"].mean())

print("Mean MW:",
      short_final_df["MolecularWeight"].mean())

print("Mean pI:",
      short_final_df["pI"].mean())

print("Mean Aromaticity:",
      short_final_df["Aromaticity"].mean())

print("Mean Instability:",
      short_final_df["InstabilityIndex"].mean())

Mean Length: 17.433464345873105
Mean MW: 2023.5649351862253
Mean pI: 10.149768181246987
Mean Aromaticity: 0.10670120673747788
Mean Instability: 30.09324413076935


In [68]:
short_final_df.head()

,Sequence,AMP_Probability,Hemolysis_Probability,Length,Final_Score,MolecularWeight,pI,Aromaticity,InstabilityIndex
5870,KILIKKLKKHIK,0.999444,0.000757,12,0.998687,1489.9764,10.699318,0.000000,26.458333
9773,KILKKIWKKIKQ,0.999614,0.001013,12,0.998601,1554.0186,10.699318,0.083333,9.541667
728,KKLKKFKFLKKAQW,0.999491,0.000973,14,0.998518,1821.3012,10.778421,0.214286,-19.642857
8464,KWLKKFKKLKQA,0.999600,0.001430,12,0.998170,1545.9551,10.699318,0.166667,5.400000
5901,KRILKWILKFKK,0.999793,0.001644,12,0.998149,1601.0766,11.388421,0.166667,41.758333


In [69]:
short_final_df.shape

(5343, 9)

In [70]:
short_final_df.to_csv(
    "final_short_amp_candidates.csv",
    index=False
)